In [2]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)

In [3]:
# Constants
IMG_HEIGHT, IMG_WIDTH = 450, 450 # Change as needed
SEED = 42
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Define base data directory
DATA_DIR = "data/2018-10000"  # Change this to your data directory

# Define all paths relative to DATA_DIR
IMAGE_DIR = os.path.join(DATA_DIR, "images")
IMAGE_RAW_DIR = os.path.join(IMAGE_DIR, "raw")
IMAGE_PROCESSED_DIR = os.path.join(IMAGE_DIR, "processed")
METADATA_DIR = os.path.join(DATA_DIR, "metadata")
METADATA_RAW_PATH = os.path.join(METADATA_DIR, "raw.csv")
METADATA_CLEANED_PATH = os.path.join(METADATA_DIR, "cleaned.csv")
TF_RECORDS_DIR = os.path.join(DATA_DIR, "tf-records")

# Create directories if they don't exist
for dir_path in [IMAGE_DIR, METADATA_DIR, TF_RECORDS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

In [5]:
def crop_and_verify_images(input_folder, output_folder):
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all image files
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    image_files = [f for f in os.listdir(input_folder) 
                  if f.lower().endswith(image_extensions)]
    
    logging.info(f"Found {len(image_files)} images to process")
    
    # Process each image
    for filename in image_files:
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)
        
        try:
            with Image.open(input_path) as img:
                # Check if image is 600x450
                if img.size != (600, 450):
                    logging.warning(f"{filename} is not 600x450 (actual size: {img.size})")
                    continue
                
                # Calculate crop box (75px from each side)
                left = 75
                top = 0
                right = 525  # 600 - 75
                bottom = 450
                
                # Crop the image
                cropped_img = img.crop((left, top, right, bottom))
                
                # Save the cropped image
                cropped_img.save(output_path)
                logging.info(f"Processed: {filename}")
                
        except Exception as e:
            logging.error(f"Error processing {filename}: {str(e)}")
    
    # Verify all images in output folder
    logging.info("\nVerifying cropped images...")
    all_correct = True
    for filename in os.listdir(output_folder):
        if filename.lower().endswith(image_extensions):
            try:
                with Image.open(os.path.join(output_folder, filename)) as img:
                    if img.size != (450, 450):
                        logging.error(f"{filename} is not 450x450 (actual size: {img.size})")
                        all_correct = False
            except Exception as e:
                logging.error(f"Error verifying {filename}: {str(e)}")
                all_correct = False
    
    if all_correct:
        logging.info("\nAll images have been successfully cropped to 450x450!")
    else:
        logging.error("\nSome images were not properly cropped. Please check the errors above.")

# Run image processing
crop_and_verify_images(IMAGE_RAW_DIR, IMAGE_PROCESSED_DIR)

2025-05-10 13:46:57,062 [INFO] Found 10015 images to process
2025-05-10 13:46:57,083 [INFO] Processed: ISIC_0030858.jpg
2025-05-10 13:46:57,085 [INFO] Processed: ISIC_0030680.jpg
2025-05-10 13:46:57,088 [INFO] Processed: ISIC_0033389.jpg
2025-05-10 13:46:57,090 [INFO] Processed: ISIC_0032097.jpg
2025-05-10 13:46:57,092 [INFO] Processed: ISIC_0032929.jpg
2025-05-10 13:46:57,094 [INFO] Processed: ISIC_0026784.jpg
2025-05-10 13:46:57,096 [INFO] Processed: ISIC_0028971.jpg
2025-05-10 13:46:57,098 [INFO] Processed: ISIC_0026948.jpg
2025-05-10 13:46:57,100 [INFO] Processed: ISIC_0026790.jpg
2025-05-10 13:46:57,102 [INFO] Processed: ISIC_0028965.jpg
2025-05-10 13:46:57,103 [INFO] Processed: ISIC_0025299.jpg
2025-05-10 13:46:57,105 [INFO] Processed: ISIC_0032083.jpg
2025-05-10 13:46:57,107 [INFO] Processed: ISIC_0024839.jpg
2025-05-10 13:46:57,108 [INFO] Processed: ISIC_0030694.jpg
2025-05-10 13:46:57,110 [INFO] Processed: ISIC_0024811.jpg
2025-05-10 13:46:57,112 [INFO] Processed: ISIC_0030864

In [4]:
def analyze_metadata(input_path, output_path):
    # Create a file handler for the summary
    summary_path = os.path.join(os.path.dirname(output_path), "summary.txt")
    file_handler = logging.FileHandler(summary_path)
    file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logging.getLogger().addHandler(file_handler)
    
    # Read the metadata
    logging.info(f"Reading metadata from {input_path}")
    df = pd.read_csv(input_path)
    logging.info(f"Original metadata shape: {df.shape}")

    # Check for missing values
    missing_values = df[['diagnosis_1', 'diagnosis_3', 'isic_id']].isnull().sum()
    logging.info("\nMissing values in key columns:")
    logging.info(missing_values)

    # Create new dataframe with only required columns
    clean_df = df[['isic_id', 'diagnosis_1', 'diagnosis_3']].copy()
    
    # Remove rows with missing values
    clean_df = clean_df.dropna()
    logging.info(f"\nAfter removing missing values: {clean_df.shape[0]} rows")

    # Filter diagnosis_1 to only include 'Benign' and 'Malignant'
    clean_df = clean_df[clean_df['diagnosis_1'].isin(['Benign', 'Malignant'])]
    logging.info(f"After removing 'Indeterminate' diagnoses: {clean_df.shape[0]} rows")

    # Show all unique values and their counts
    logging.info("\nFinal unique values in diagnosis_1:")
    logging.info(clean_df['diagnosis_1'].value_counts())
    
    logging.info("\nFinal unique values in diagnosis_3:")
    logging.info(clean_df['diagnosis_3'].value_counts())

    # Save the cleaned metadata
    clean_df.to_csv(output_path, index=False)
    logging.info(f"\nSaved cleaned metadata to {output_path}")
    
    # Remove the file handler to avoid duplicate logging
    logging.getLogger().removeHandler(file_handler)
    
    return clean_df

# Run metadata analysis and cleaning
cleaned_df = analyze_metadata(METADATA_RAW_PATH, METADATA_CLEANED_PATH)

2025-05-10 14:24:05,610 [INFO] Reading metadata from data/2018-10000/metadata/raw.csv
2025-05-10 14:24:05,637 [INFO] Original metadata shape: (10015, 16)
2025-05-10 14:24:05,639 [INFO] 
Missing values in key columns:
2025-05-10 14:24:05,639 [INFO] diagnosis_1      0
diagnosis_3    142
isic_id          0
dtype: int64
2025-05-10 14:24:05,642 [INFO] 
After removing missing values: 9873 rows
2025-05-10 14:24:05,643 [INFO] After removing 'Indeterminate' diagnoses: 9743 rows
2025-05-10 14:24:05,643 [INFO] 
Final unique values in diagnosis_1:
2025-05-10 14:24:05,644 [INFO] diagnosis_1
Benign       7919
Malignant    1824
Name: count, dtype: int64
2025-05-10 14:24:05,644 [INFO] 
Final unique values in diagnosis_3:
2025-05-10 14:24:05,645 [INFO] diagnosis_3
Nevus                           6705
Melanoma, NOS                   1113
Pigmented benign keratosis      1099
Basal cell carcinoma             514
Squamous cell carcinoma, NOS     197
Dermatofibroma                   115
Name: count, dtype: 

In [5]:
def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def _float_feature(value):
    return tf.train.Feature(float_list=tf.train.FloatList(value=[value]))

def serialize_example(image_string, binary_label, diagnosis_label, binary_weight, diagnosis_weight):
    feature = {
        "image": _bytes_feature(image_string),
        "binary_label": _int64_feature(binary_label),
        "diagnosis_label": _int64_feature(diagnosis_label),
        "binary_weight": _float_feature(binary_weight),
        "diagnosis_weight": _float_feature(diagnosis_weight),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature)).SerializeToString()

def write_tfrecord(df_subset, image_dir, output_path):
    count = 0
    with tf.io.TFRecordWriter(output_path) as writer:
        for idx, row in df_subset.iterrows():
            image_path = os.path.join(image_dir, f"{row['isic_id']}.jpg")
            try:
                image_data = tf.io.read_file(image_path)
                example = serialize_example(
                    image_data.numpy(),
                    int(row["binary_label"]),
                    int(row["diagnosis_label"]),
                    float(row["binary_weight"]),
                    float(row["diagnosis_weight"]),
                )
                writer.write(example)
                count += 1
            except Exception as e:
                logging.warning(f"Skipping {row['isic_id']}: {e}")
    logging.info(f"Wrote {count} records to {output_path}")

def create_tf_records(df, image_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    # Dynamically create label maps from the CSV
    binary_classes = sorted(df["diagnosis_1"].dropna().unique())
    binary_map = {name: idx for idx, name in enumerate(binary_classes)}
    diagnosis_classes = sorted(df["diagnosis_3"].dropna().unique())
    diagnosis_map = {name: idx for idx, name in enumerate(diagnosis_classes)}
    logging.info(f"Binary label mapping: {binary_map}")
    logging.info(f"Diagnosis label mapping: {diagnosis_map}")

    # Map string labels to integers
    df["binary_label"] = df["diagnosis_1"].map(binary_map)
    df["diagnosis_label"] = df["diagnosis_3"].map(diagnosis_map)

    if df["binary_label"].isnull().any() or df["diagnosis_label"].isnull().any():
        raise ValueError("Unmapped labels found. Check your CSV and label maps.")

    # Compute class weights
    binary_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.sort(df['binary_label'].unique()),
        y=df['binary_label']
    )
    binary_class_weight = dict(zip(np.sort(df['binary_label'].unique()), binary_weights))

    diagnosis_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.sort(df['diagnosis_label'].unique()),
        y=df['diagnosis_label']
    )
    diagnosis_class_weight = dict(zip(np.sort(df['diagnosis_label'].unique()), diagnosis_weights))

    df['binary_weight'] = df['binary_label'].map(binary_class_weight)
    df['diagnosis_weight'] = df['diagnosis_label'].map(diagnosis_class_weight)

    df["stratify"] = (
        df["binary_label"].astype(str) + "_" + df["diagnosis_label"].astype(str)
    )

    # Split dataset
    train_val_df, test_df = train_test_split(
        df, test_size=TEST_SPLIT, stratify=df["stratify"], random_state=SEED
    )

    val_rel_split = VAL_SPLIT / (1 - TEST_SPLIT)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_rel_split,
        stratify=train_val_df["stratify"],
        random_state=SEED,
    )

    logging.info(
        f"Total: {len(df)} | Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}"
    )

    # Write TFRecords
    write_tfrecord(
        train_df, image_dir, os.path.join(output_dir, "train.tfrecord")
    )
    write_tfrecord(
        val_df, image_dir, os.path.join(output_dir, "val.tfrecord")
    )
    write_tfrecord(
        test_df, image_dir, os.path.join(output_dir, "test.tfrecord")
    )

    logging.info("TFRecord conversion completed successfully.")

# Run TFRecord creation
create_tf_records(cleaned_df, IMAGE_PROCESSED_DIR, TF_RECORDS_DIR)

2025-05-10 14:24:08,951 [INFO] Binary label mapping: {'Benign': 0, 'Malignant': 1}
2025-05-10 14:24:08,952 [INFO] Diagnosis label mapping: {'Basal cell carcinoma': 0, 'Dermatofibroma': 1, 'Melanoma, NOS': 2, 'Nevus': 3, 'Pigmented benign keratosis': 4, 'Squamous cell carcinoma, NOS': 5}
2025-05-10 14:24:08,973 [INFO] Total: 9743 | Train: 6819 | Val: 1462 | Test: 1462
2025-05-10 14:24:09,617 [INFO] Wrote 6819 records to data/2018-10000/tf-records/train.tfrecord
2025-05-10 14:24:09,747 [INFO] Wrote 1462 records to data/2018-10000/tf-records/val.tfrecord
2025-05-10 14:24:09,879 [INFO] Wrote 1462 records to data/2018-10000/tf-records/test.tfrecord
2025-05-10 14:24:09,879 [INFO] TFRecord conversion completed successfully.
